# Notebook 02 — Data Preparation

**Phase CRISP-DM : 3 sur 6**

Objectif : nettoyer, enrichir et structurer les données pour la phase de modélisation.

### Plan de traitement
1. Nettoyage (doublons, prix aberrants, titres vides)
2. Extraction d'entités depuis le titre (type de bien, type de transaction, quartier)
3. Normalisation de la ville
4. Construction de la colonne `texte_modele` (features texte concaténées)
5. Sauvegarde du dataset préparé

In [1]:
import re
import unicodedata
import pandas as pd
import numpy as np

df = pd.read_csv("../data/processed/annonces_immobilier.csv")
print(f"Avant nettoyage : {df.shape}")

Avant nettoyage : (255, 10)


## 1. Nettoyage

In [2]:
# 1.1 Supprimer les titres manquants (ou les remplacer par le début du document)
df["titre"] = df["titre"].fillna(df["document"].str.slice(0, 80))

# 1.2 Supprimer les doublons stricts (même id ou même titre+prix+ville)
df = df.drop_duplicates(subset=["id"]).copy()
df = df.drop_duplicates(subset=["titre", "prix", "ville"]).copy()

# 1.3 Marquer les prix suspects : 0 ou < 100 CFA
df["prix_valide"] = (df["prix"] >= 100)

# 1.4 Statistiques après nettoyage
print(f"Après déduplication : {df.shape}")
print(f"Annonces avec prix valide : {df['prix_valide'].sum()}")

Après déduplication : (255, 11)
Annonces avec prix valide : 177


## 2. Extraction d'entités depuis le titre

In [3]:
def strip_accents(text: str) -> str:
    nfkd = unicodedata.normalize("NFD", str(text))
    return "".join(c for c in nfkd if not unicodedata.combining(c)).lower()

df["titre_norm"] = df["titre"].map(strip_accents)
df["doc_norm"]   = df["document"].fillna("").map(strip_accents)
df["corpus"]     = df["titre_norm"] + " " + df["doc_norm"]
df["corpus"].head(3).tolist()

['chambre salon nouvelle construction a louer a agoe zossime 00228 93818957 titre: chambre salon nouvelle construction a louer a agoe zossime 00228 93818957. description: . prix: 35000.0 cfa. ville: togo.',
 'a louer chambre salon wc douche ( sanguera klikame) titre: a louer chambre salon wc douche ( sanguera klikame). description: . prix: 40000.0 cfa. ville: togo.',
 'a louer deux chambre salon agoe daliko titre: a louer deux chambre salon agoe daliko. description: . prix: 40000.0 cfa. ville: togo.']

In [4]:
# 2.1 Type de transaction : location / vente / terrain / autre
def detect_transaction(text: str) -> str:
    if re.search(r"\b(louer|location|a louer|a\s*louer|loue)\b", text):
        return "location"
    if re.search(r"\b(vendre|vente|a vendre|a\s*vendre|vends)\b", text):
        return "vente"
    if re.search(r"\b(terrain|lot|parcelle|titre foncier)\b", text):
        return "terrain"
    return "autre"

df["type_transaction"] = df["corpus"].map(detect_transaction)
df["type_transaction"].value_counts()

type_transaction
location    77
autre       67
vente       57
terrain     54
Name: count, dtype: int64

In [5]:
# 2.2 Type de bien : chambre-salon / appartement / maison / villa / terrain / autre
def detect_type_bien(text: str) -> str:
    if re.search(r"chambre\s*salon|studio|chambre\s+s", text):
        return "chambre_salon"
    if re.search(r"appartement|appart\b", text):
        return "appartement"
    if re.search(r"villa|duplex", text):
        return "villa"
    if re.search(r"\bmaison\b", text):
        return "maison"
    if re.search(r"terrain|lot|parcelle", text):
        return "terrain"
    if re.search(r"bureau|boutique|magasin|entrepot", text):
        return "commerce"
    return "autre"

df["type_bien"] = df["corpus"].map(detect_type_bien)
df["type_bien"].value_counts()

type_bien
terrain          76
chambre_salon    66
villa            58
appartement      30
autre            18
maison            4
commerce          3
Name: count, dtype: int64

In [6]:
# 2.3 Quartier/ville détaillée (regex sur liste connue)
QUARTIERS = {
    "lome": "Lomé",
    "agoe": "Agoè",
    "adidogome": "Adidogomé",
    "kegue": "Kégué",
    "baguida": "Baguida",
    "tokoin": "Tokoin",
    "avedji": "Avédji",
    "cacaveli": "Cacavéli",
    "djidjole": "Djidjolé",
    "kodjoviakope": "Kodjoviakopé",
    "totsi": "Totsi",
    "nyekonakpoe": "Nyékonakpoè",
    "sanguera": "Sanguéra",
    "legbassito": "Legbassito",
    "hedzranawoe": "Hédzranawoé",
    "akodessewa": "Akodessewa",
    "kpogan": "Kpogan",
    "djagble": "Djagblé",
    "avepozo": "Avépozo",
    "tsevie": "Tsévié",
    "kpalime": "Kpalimé",
    "tabligbo": "Tabligbo",
    "afagnan": "Afagnan",
    "aneho": "Aného",
}

def detect_quartier(text: str) -> str:
    for key, label in QUARTIERS.items():
        if key in text:
            return label
    return "Inconnu"

df["quartier"] = df["corpus"].map(detect_quartier)
df["quartier"].value_counts().head(10)

quartier
Inconnu      115
Lomé          67
Agoè          21
Adidogomé      9
Avédji         5
Baguida        5
Totsi          4
Kpogan         4
Tokoin         3
Djidjolé       3
Name: count, dtype: int64

## 3. Normalisation de la colonne ville

In [7]:
# Si la ville est "Togo" (générique) et qu'on a détecté un quartier → on remplace
def normaliser_ville(row):
    if row["quartier"] != "Inconnu":
        if row["quartier"] in {"Tsévié", "Kpalimé", "Tabligbo", "Afagnan", "Aného"}:
            return row["quartier"]
        return "Lomé"  # tous les autres quartiers sont dans l'agglo lomoise
    if row["ville"] in {"Togo", None}:
        return "Inconnu"
    return row["ville"]

df["ville_norm"] = df.apply(normaliser_ville, axis=1)
df["ville_norm"].value_counts()

ville_norm
Lomé        135
Inconnu     115
Kpalimé       2
Aného         1
Tabligbo      1
Afagnan       1
Name: count, dtype: int64

## 4. Construction de la feature texte pour le TF-IDF

In [8]:
# On concatène titre + type_bien + transaction + ville + quartier pour donner du poids aux features métier
df["texte_modele"] = (
    df["titre_norm"].fillna("") + " "
    + df["type_bien"].fillna("") + " "
    + df["type_transaction"].fillna("") + " "
    + df["ville_norm"].fillna("").map(strip_accents) + " "
    + df["quartier"].fillna("").map(strip_accents)
).str.strip()

df["texte_modele"].head(5).tolist()

['chambre salon nouvelle construction a louer a agoe zossime 00228 93818957 chambre_salon location lome agoe',
 'a louer chambre salon wc douche ( sanguera klikame) chambre_salon location lome sanguera',
 'a louer deux chambre salon agoe daliko chambre_salon location lome agoe',
 'chambre salon interne a louer chambre_salon location inconnu inconnu',
 'chambre salon wcd cuisine interne chambre_salon autre inconnu inconnu']

## 5. Sauvegarde du dataset préparé

In [9]:
cols_keep = [
    "id", "titre", "prix", "prix_valide",
    "ville_norm", "quartier", "source",
    "type_transaction", "type_bien",
    "texte_modele", "lien", "image_url",
]
df_clean = df[cols_keep].copy()
df_clean.to_csv("../data/processed/annonces_clean.csv", index=False)
print(f"Dataset nettoyé sauvegardé : {df_clean.shape}")
df_clean.head(3)

Dataset nettoyé sauvegardé : (255, 12)


,id,titre,prix,prix_valide,ville_norm,quartier,source,type_transaction,type_bien,texte_modele,lien,image_url
0,fb_69a97f6185f1dd70f9aef290,CHAMBRE SALON NOUVELLE CONSTRUCTION À LOUER À ...,35000.0,True,Lomé,Agoè,Facebook,location,chambre_salon,chambre salon nouvelle construction a louer a ...,https://www.facebook.com/marketplace/item/9123...,https://scontent-cph2-1.xx.fbcdn.net/v/t39.308...
1,fb_69a97f6185f1dd70f9aef291,A louer Chambre salon wc douche ( SANGUERA KLI...,40000.0,True,Lomé,Sanguéra,Facebook,location,chambre_salon,a louer chambre salon wc douche ( sanguera kli...,https://www.facebook.com/marketplace/item/2684...,https://scontent-cph2-1.xx.fbcdn.net/v/t39.847...
2,fb_69a97f6185f1dd70f9aef292,A louer deux chambre salon agoè daliko,40000.0,True,Lomé,Agoè,Facebook,location,chambre_salon,a louer deux chambre salon agoe daliko chambre...,https://www.facebook.com/marketplace/item/9504...,https://scontent-cph2-1.xx.fbcdn.net/v/t39.847...


### Conclusion — livrables de la phase Data Preparation
- Dataset nettoyé : `data/processed/annonces_clean.csv`
- 4 nouvelles features métier : `type_transaction`, `type_bien`, `quartier`, `ville_norm`
- Colonne `texte_modele` prête pour le TF-IDF
- Volume conservé : ~98% des lignes (seulement les doublons stricts sont retirés)

➡️ Passage à la phase 4 : **Modeling** (notebook 03).